# Pipeline: Download → Normalize → Analyze

Orchestrator notebook that runs the full data pipeline using modules from `src/`.

Each step can be skipped if the data already exists.

In [3]:
import os
import sys

# Install dependencies
!pip install -q pyspark==3.4.2 apache-sedona==1.6.1 requests folium

## Step 1: Download Raw Data

Downloads NY crime data (Socrata API), police stations, and transport stations (Overpass API).

Skips download if files already exist.

In [5]:
from src.config import NY_CRIMES_RAW, NY_POLICE_STATIONS, NY_TRANSPORT_STATIONS
from src.download import download_ny_crimes, download_police_stations, download_transport_stations

# Check what already exists
crimes_exist = os.path.isdir(NY_CRIMES_RAW) and len(os.listdir(NY_CRIMES_RAW)) > 0
police_exist = os.path.isfile(NY_POLICE_STATIONS)
transport_exist = os.path.isfile(NY_TRANSPORT_STATIONS)

if crimes_exist:
    print(f"Crime CSVs already exist in {NY_CRIMES_RAW} — skipping download")
else:
    download_ny_crimes()

if police_exist:
    print(f"Police stations already exist at {NY_POLICE_STATIONS} — skipping download")
else:
    download_police_stations()

if transport_exist:
    print(f"Transport stations already exist at {NY_TRANSPORT_STATIONS} — skipping download")
else:
    download_transport_stations()

print("\nStep 1 complete.")

Crime CSVs already exist in data/new_york/crimes — skipping download
Police stations already exist at data/new_york/police_stations/new_york_police_stations.csv — skipping download
Transport stations already exist at data/new_york/transport_stations/new_york_transport_stations.csv — skipping download

Step 1 complete.


## Step 2: Normalize Crime Data

Renames columns, casts types, filters invalid coordinates, and saves as partitioned parquet.

Skips if normalized parquet already exists.

In [6]:
from src.config import NY_CRIMES_NORMALIZED
from src.spark_session import get_or_create_spark
from src.normalize import normalize_ny_crimes

spark = get_or_create_spark()

if os.path.isdir(NY_CRIMES_NORMALIZED):
    print(f"Normalized parquet already exists at {NY_CRIMES_NORMALIZED} — skipping normalization")
    df_crimes = spark.read.parquet(NY_CRIMES_NORMALIZED)
else:
    df_crimes = normalize_ny_crimes(spark)

print(f"Crimes loaded: {df_crimes.count():,} records")
df_crimes.printSchema()
df_crimes.show(5)

Normalized parquet already exists at data/new_york/crimes_normalized/parquet — skipping normalization
Crimes loaded: 3,617 records
root
 |-- complaint_id: string (nullable = true)
 |-- precinct_code: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- crime_start_date: date (nullable = true)
 |-- crime_start_time: string (nullable = true)
 |-- crime_end_date: date (nullable = true)
 |-- crime_end_time: string (nullable = true)
 |-- report_date: date (nullable = true)
 |-- crime_status: string (nullable = true)
 |-- jurisdiction_description: string (nullable = true)
 |-- offense_level: string (nullable = true)
 |-- offense_description: string (nullable = true)
 |-- internal_description: string (nullable = true)
 |-- premise_type: string (nullable = true)
 |-- location_type: string (nullable = true)
 |-- patrol_borough: string (nullable = true)
 |-- suspect_age_group: string (nullable = true)
 |-- suspect_race: string (nullable = true)
 |-- suspect_sex: string (nullable

## Step 3: Spatial Analysis

Loads police/transport stations, computes distances from each crime to nearest station, and categorizes proximity.

In [7]:
from sedona.register import SedonaRegistrator
from pyspark.sql.functions import col, expr

SedonaRegistrator.registerAll(spark)

# Load station data
df_police = spark.read.option("header", True).csv(NY_POLICE_STATIONS)
df_transport = spark.read.option("header", True).csv(NY_TRANSPORT_STATIONS)

print(f"Police stations: {df_police.count()}")
print(f"Transport stations: {df_transport.count()}")

/tmp/ipykernel_1917/1323539997.py:4: DeprecationWarning: Call to deprecated function registerAll (Deprecated since 1.4.1, use SedonaContext.create() instead.).
  SedonaRegistrator.registerAll(spark)


Police stations: 1805
Transport stations: 5870


In [8]:
# Create geometry columns using Sedona
df_crimes_geo = df_crimes.withColumn(
    "geometry", expr("ST_Point(CAST(longitude AS DOUBLE), CAST(latitude AS DOUBLE))")
)
df_police_geo = df_police.withColumn(
    "geometry", expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
)
df_transport_geo = df_transport.withColumn(
    "geometry", expr("ST_Point(CAST(lon AS DOUBLE), CAST(lat AS DOUBLE))")
)

In [9]:
from src.analyze import compute_spatial_distances, categorize_proximity

# Compute distances to nearest police and transport stations
df_distances = compute_spatial_distances(spark, df_crimes_geo, df_police_geo, df_transport_geo)

# Categorize proximity
df_categorized = categorize_proximity(df_distances)

print("Proximity categorization:")
df_categorized.select("offense_description", "dist_to_nearest_police_km", "near_police",
                      "dist_to_nearest_transport_km", "near_transport").show(10)

Proximity categorization:
+--------------------+-------------------------+-------------+----------------------------+----------------+
| offense_description|dist_to_nearest_police_km|  near_police|dist_to_nearest_transport_km|  near_transport|
+--------------------+-------------------------+-------------+----------------------------+----------------+
|       PETIT LARCENY|       3.1029883270405194|LEJOS_POLICIA|          0.9384392058635335|LEJOS_TRANSPORTE|
|GRAND LARCENY OF ...|        3.214830167609356|LEJOS_POLICIA|          1.4486626345875588|LEJOS_TRANSPORTE|
|                RAPE|      0.03280195103492409|CERCA_POLICIA|          1.3013071452523033|LEJOS_TRANSPORTE|
|       PETIT LARCENY|       1.9553849180270186|LEJOS_POLICIA|           5.824489925646364|LEJOS_TRANSPORTE|
|CRIMINAL MISCHIEF...|        1.563165410170962|LEJOS_POLICIA|          3.4174376355656926|LEJOS_TRANSPORTE|
|MISCELLANEOUS PEN...|        1.646048689366564|LEJOS_POLICIA|          0.5636634628304705|LEJOS_TRANS

## Step 4: Temporal Analysis

Enrich crime data with time features (hour, day of week, month, time slot) and analyze temporal patterns.

In [10]:
from src.analyze import add_temporal_columns, analyze_temporal_patterns, analyze_crime_type_by_time_slot

# Enrich crimes with temporal features
df_temporal = add_temporal_columns(df_crimes)

# Analyze patterns
crimes_by_hour, crimes_by_day, crimes_by_month, crimes_by_slot = analyze_temporal_patterns(df_temporal)

print("Crimes by hour of day:")
crimes_by_hour.show(24)

print("Crimes by day of week (1=Domingo, 2=Lunes, ..., 7=Sabado):")
crimes_by_day.show(7)

print("Crimes by month:")
crimes_by_month.show(12)

print("Crimes by time slot:")
crimes_by_slot.show()

print("Top crime types by time slot:")
analyze_crime_type_by_time_slot(df_temporal).show(30, truncate=False)

Crimes by hour of day:
+----------+------------+
|crime_hour|total_crimes|
+----------+------------+
|         0|         575|
|         1|          72|
|         2|          54|
|         3|          39|
|         4|          33|
|         5|          32|
|         6|          40|
|         7|          83|
|         8|         244|
|         9|         221|
|        10|         170|
|        11|         140|
|        12|         416|
|        13|         143|
|        14|         175|
|        15|         201|
|        16|         150|
|        17|         143|
|        18|         144|
|        19|         116|
|        20|         150|
|        21|          95|
|        22|          96|
|        23|          85|
+----------+------------+

Crimes by day of week (1=Domingo, 2=Lunes, ..., 7=Sabado):
+-----------------+------------+
|crime_day_of_week|total_crimes|
+-----------------+------------+
|                1|         457|
|                2|         544|
|                3|     

## Step 5: FP-Growth (Association Rules)

In [11]:
from src.analyze import run_fp_growth

freq_itemsets, rules = run_fp_growth(df_categorized)

print("Frequent Itemsets:")
freq_itemsets.orderBy("freq", ascending=False).show(20, truncate=False)

print("Association Rules:")
rules.orderBy("confidence", ascending=False).show(20, truncate=False)

Frequent Itemsets:
+-----------------------------------------------------+----+
|items                                                |freq|
+-----------------------------------------------------+----+
|[LEJOS_POLICIA]                                      |2748|
|[LEJOS_TRANSPORTE]                                   |2083|
|[LEJOS_TRANSPORTE, LEJOS_POLICIA]                    |1855|
|[CERCA_TRANSPORTE]                                   |1534|
|[CERCA_TRANSPORTE, LEJOS_POLICIA]                    |893 |
|[CERCA_POLICIA]                                      |869 |
|[TIPO=GRAND LARCENY]                                 |704 |
|[CERCA_POLICIA, CERCA_TRANSPORTE]                    |641 |
|[TIPO=GRAND LARCENY, LEJOS_POLICIA]                  |579 |
|[TIPO=PETIT LARCENY]                                 |557 |
|[TIPO=HARRASSMENT 2]                                 |496 |
|[TIPO=PETIT LARCENY, LEJOS_POLICIA]                  |460 |
|[TIPO=HARRASSMENT 2, LEJOS_POLICIA]                  |449 |
|[TIP

## Step 6: Expanded FP-Growth (with Time Slot and Demographics)

Adds time slot (MADRUGADA/MANANA/TARDE/NOCHE) and victim demographics to the association rules for richer pattern discovery.

In [12]:
from src.analyze import run_fp_growth_expanded

# Join spatial categorization with temporal + demographic features
df_expanded = (
    df_categorized
    .join(
        df_temporal.select("complaint_id", "time_slot"),
        on="complaint_id",
        how="inner"
    )
    .join(
        df_crimes.select("complaint_id", "victim_sex", "victim_age_group"),
        on="complaint_id",
        how="inner"
    )
)

freq_exp, rules_exp = run_fp_growth_expanded(df_expanded)

print("Expanded Frequent Itemsets (with time slot + demographics):")
freq_exp.orderBy("freq", ascending=False).show(20, truncate=False)

print("Expanded Association Rules:")
rules_exp.orderBy("confidence", ascending=False).show(20, truncate=False)

Expanded Frequent Itemsets (with time slot + demographics):
+-------------------------------------------------+----+
|items                                            |freq|
+-------------------------------------------------+----+
|[LEJOS_POLICIA]                                  |2748|
|[LEJOS_TRANSPORTE]                               |2083|
|[LEJOS_TRANSPORTE, LEJOS_POLICIA]                |1855|
|[VICTIMA_SEXO=F]                                 |1712|
|[CERCA_TRANSPORTE]                               |1534|
|[HORARIO=TARDE]                                  |1228|
|[VICTIMA_SEXO=F, LEJOS_POLICIA]                  |1187|
|[VICTIMA_EDAD=25-44]                             |1173|
|[VICTIMA_SEXO=M]                                 |1094|
|[HORARIO=TARDE, LEJOS_POLICIA]                   |989 |
|[VICTIMA_SEXO=F, LEJOS_TRANSPORTE]               |968 |
|[VICTIMA_EDAD=25-44, LEJOS_POLICIA]              |923 |
|[VICTIMA_SEXO=M, LEJOS_POLICIA]                  |919 |
|[HORARIO=MANANA]           

## Step 7: Offense Level Breakdown

Analyzes crime severity (FELONY / MISDEMEANOR / VIOLATION) overall, by borough, and by time slot.

In [13]:
from src.analyze import analyze_offense_level

# Offense level needs temporal columns for time_slot breakdown
level_counts, level_by_borough, level_by_time_slot = analyze_offense_level(df_temporal)

print("Overall offense level distribution:")
level_counts.show()

print("Offense level by borough:")
level_by_borough.show(20, truncate=False)

print("Offense level by time slot:")
level_by_time_slot.show(20, truncate=False)

Overall offense level distribution:
+-------------+------------+
|offense_level|total_crimes|
+-------------+------------+
|       FELONY|        1570|
|  MISDEMEANOR|        1543|
|    VIOLATION|         504|
+-------------+------------+

Offense level by borough:
+-------------+-------------+------------+
|borough      |offense_level|total_crimes|
+-------------+-------------+------------+
|BRONX        |FELONY       |170         |
|BRONX        |MISDEMEANOR  |114         |
|BRONX        |VIOLATION    |27          |
|BROOKLYN     |FELONY       |342         |
|BROOKLYN     |MISDEMEANOR  |304         |
|BROOKLYN     |VIOLATION    |74          |
|MANHATTAN    |FELONY       |366         |
|MANHATTAN    |MISDEMEANOR  |252         |
|MANHATTAN    |VIOLATION    |62          |
|QUEENS       |FELONY       |209         |
|QUEENS       |MISDEMEANOR  |131         |
|QUEENS       |VIOLATION    |49          |
|STATEN ISLAND|MISDEMEANOR  |726         |
|STATEN ISLAND|FELONY       |483         |
|ST

## Step 8: Premise Type Analysis

Identifies which locations (STREET, RESIDENCE, CHAIN STORE, etc.) concentrate the most crimes and which crime types are associated with each premise.

In [14]:
from src.analyze import analyze_premise_type

top_premises, premise_by_crime_type = analyze_premise_type(df_crimes)

print("Top 20 premise types by crime count:")
top_premises.show(20, truncate=False)

print("Top 30 premise + crime type combinations:")
premise_by_crime_type.show(30, truncate=False)

Top 20 premise types by crime count:
+---------------------------+------------+
|premise_type               |total_crimes|
+---------------------------+------------+
|RESIDENCE-HOUSE            |1026        |
|RESIDENCE - APT. HOUSE     |938         |
|STREET                     |489         |
|RESIDENCE - PUBLIC HOUSING |170         |
|COMMERCIAL BUILDING        |155         |
|CHAIN STORE                |121         |
|OTHER                      |90          |
|PUBLIC BUILDING            |54          |
|DEPARTMENT STORE           |51          |
|PUBLIC SCHOOL              |42          |
|HOMELESS SHELTER           |32          |
|RESTAURANT/DINER           |29          |
|BANK                       |24          |
|TRANSIT - NYC SUBWAY       |24          |
|GROCERY/BODEGA             |20          |
|PARKING LOT/GARAGE (PUBLIC)|19          |
|DRUG STORE                 |19          |
|SMALL MERCHANT             |18          |
|MAILBOX OUTSIDE            |17          |
|BAR/NIGHT CLUB  

## Step 9: Anomaly Detection

Uses z-score method to flag months where crime counts per borough or per crime type deviate significantly (>2 std deviations) from their historical mean.

In [15]:
from src.analyze import detect_anomalies_by_borough_month, detect_anomalies_by_crime_type_month

# Anomalies by borough
borough_scores, borough_anomalies = detect_anomalies_by_borough_month(df_crimes)

print("Monthly crime scores by borough (all months):")
borough_scores.show(40, truncate=False)

print("Anomalous months by borough (z-score > 2.0):")
borough_anomalies.select(
    "borough", "crime_year", "crime_month", "crime_count", "z_score"
).show(30, truncate=False)

# Anomalies by crime type
type_scores, type_anomalies = detect_anomalies_by_crime_type_month(df_crimes)

print("Anomalous months by crime type (z-score > 2.0):")
type_anomalies.select(
    "offense_description", "crime_year", "crime_month", "crime_count", "z_score"
).show(30, truncate=False)

Monthly crime scores by borough (all months):
+-------+----------+-----------+-----------+-----------------+-----------------+-------+----------+
|borough|crime_year|crime_month|crime_count|borough_mean     |borough_stddev   |z_score|is_anomaly|
+-------+----------+-----------+-----------+-----------------+-----------------+-------+----------+
|BRONX  |2020      |1          |6          |6.617021276595745|5.269077930593539|-0.12  |false     |
|BRONX  |2020      |2          |3          |6.617021276595745|5.269077930593539|-0.69  |false     |
|BRONX  |2020      |3          |4          |6.617021276595745|5.269077930593539|-0.5   |false     |
|BRONX  |2020      |5          |4          |6.617021276595745|5.269077930593539|-0.5   |false     |
|BRONX  |2020      |6          |3          |6.617021276595745|5.269077930593539|-0.69  |false     |
|BRONX  |2020      |7          |7          |6.617021276595745|5.269077930593539|0.07   |false     |
|BRONX  |2020      |8          |3          |6.61702127

## Step 10: K-Means Clustering

In [16]:
from src.analyze import run_kmeans

model, predictions = run_kmeans(df_crimes)

print("Cluster centers:")
for i, center in enumerate(model.clusterCenters()):
    print(f"  Cluster {i}: lon={center[0]:.4f}, lat={center[1]:.4f}")

print(f"\nCrimes per cluster:")
predictions.groupBy("prediction").count().orderBy("prediction").show()

Cluster centers:
  Cluster 0: lon=-73.9791, lat=40.7556
  Cluster 1: lon=-74.1900, lat=40.5461
  Cluster 2: lon=-73.7701, lat=40.6878
  Cluster 3: lon=-73.9999, lat=40.6204
  Cluster 4: lon=-73.9184, lat=40.6701
  Cluster 5: lon=-74.1268, lat=40.6123
  Cluster 6: lon=-73.9996, lat=40.7122
  Cluster 7: lon=-73.8691, lat=40.8563
  Cluster 8: lon=-73.9269, lat=40.8155
  Cluster 9: lon=-73.8532, lat=40.7353

Crimes per cluster:
+----------+-----+
|prediction|count|
+----------+-----+
|         0|  289|
|         1|  440|
|         2|  134|
|         3|  355|
|         4|  273|
|         5| 1061|
|         6|  432|
|         7|  204|
|         8|  244|
|         9|  185|
+----------+-----+



## Step 11: Visualization

In [17]:
import folium

# Create map centered on New York
ny_map = folium.Map(location=[40.7128, -74.0060], zoom_start=11)

# Add cluster centers
for i, center in enumerate(model.clusterCenters()):
    folium.Marker(
        location=[center[1], center[0]],  # lat, lon
        popup=f"Cluster {i}",
        icon=folium.Icon(color="red", icon="info-sign"),
    ).add_to(ny_map)

# Add police stations
police_rows = df_police.collect()
for row in police_rows:
    if row["lat"] and row["lon"]:
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=4,
            color="blue",
            fill=True,
            popup=row["name"] or "Police Station",
        ).add_to(ny_map)

# Add transport stations (sample to avoid overloading map)
transport_rows = df_transport.limit(200).collect()
for row in transport_rows:
    if row["lat"] and row["lon"]:
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=3,
            color="green",
            fill=True,
            popup=row["name"] or "Transport Station",
        ).add_to(ny_map)

ny_map

In [18]:
spark.stop()
print("Pipeline complete.")

Pipeline complete.
